In [1]:
import pandas as pd

eventos = pd.read_csv("../data/eventos_todos_completo.csv")
playbooks = pd.read_csv("../data/playbooks.csv")

print(eventos["indicador"].value_counts())
print(playbooks)

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64
     indicador ativo_alvo direcao_se_surpresa_positiva  \
0      CPI_EUA        SPY                       vender   
1      IPCA_BR        EWZ                       vender   
2     Selic_BR      BRL=X                       vender   
3  Payroll_EUA        SPY                       vender   

  direcao_se_surpresa_negativa  
0                      comprar  
1                      comprar  
2                      comprar  
3                      comprar  


In [2]:
LIMIAR_SURPRESA = 1.0

eventos["limiar_ian_indicador"] = eventos.groupby("indicador")["IAN"].transform(lambda x: x.quantile(0.75))
eventos["opera"] = (eventos["surpresa_zscore"].abs() > LIMIAR_SURPRESA) & (eventos["IAN"] > eventos["limiar_ian_indicador"])

print(eventos.groupby("indicador")["opera"].sum())
print(f"\nTotal geral de operações: {eventos['opera'].sum()}")

indicador
CPI_EUA        5
IPCA_BR        3
Payroll_EUA    3
Selic_BR       2
Name: opera, dtype: int64

Total geral de operações: 13


In [3]:
def determinar_direcao(row, playbooks):
    if not row["opera"]:
        return None
    regra = playbooks[playbooks["indicador"] == row["indicador"]].iloc[0]
    if row["surpresa_zscore"] > 0:
        return regra["direcao_se_surpresa_positiva"]
    else:
        return regra["direcao_se_surpresa_negativa"]

eventos["direcao"] = eventos.apply(lambda row: determinar_direcao(row, playbooks), axis=1)

In [4]:
eventos["tamanho_posicao"] = eventos["IAN"] * (1 + eventos["ICE"])
eventos.loc[~eventos["opera"], "tamanho_posicao"] = 0

In [5]:
operacoes = eventos[eventos["opera"]]
print(operacoes[["data", "indicador", "surpresa_zscore", "IAN", "direcao", "tamanho_posicao"]])

                    data    indicador  surpresa_zscore       IAN  direcao  \
1             2023-04-12      CPI_EUA        -1.007506  0.356322  comprar   
13            2024-04-10      CPI_EUA         1.007506  0.333333   vender   
33            2026-02-13      CPI_EUA        -1.007506  0.459770  comprar   
35            2026-04-10      CPI_EUA        -1.007506  0.333333  comprar   
38            2026-07-14      CPI_EUA        -3.022518  0.344828  comprar   
63   2025-02-11 00:00:00      IPCA_BR        -1.237121  0.526882  comprar   
64   2025-03-12 00:00:00      IPCA_BR         4.182419  0.591398   vender   
65   2025-04-11 00:00:00      IPCA_BR         1.141957  0.333333   vender   
73   2023-08-03 00:00:00     Selic_BR        -2.402829  0.771429  comprar   
82   2024-12-12 00:00:00     Selic_BR         2.402829  0.800000   vender   
128           2026-03-06  Payroll_EUA        -1.745893  0.762887  comprar   
129           2026-04-03  Payroll_EUA         1.315240  0.639175   vender   

In [6]:
eventos.to_csv("../data/eventos_com_decisao.csv", index=False)

In [7]:
import pandas as pd

playbooks = pd.read_csv("../data/playbooks.csv")

nomes_indicadores = {
    "CPI_EUA": "CPI (EUA)",
    "IPCA_BR": "IPCA (Brasil)",
    "Selic_BR": "Selic (Brasil)",
    "Payroll_EUA": "Payroll (EUA)"
}

nomes_ativos = {
    "SPY": "ETF de índice de ações (SPY)",
    "EWZ": "ETF da bolsa brasileira (EWZ)",
    "BRL=X": "câmbio USD/BRL"
}

linhas = []
for _, row in playbooks.iterrows():
    indicador_nome = nomes_indicadores.get(row["indicador"], row["indicador"])
    ativo_nome = nomes_ativos.get(row["ativo_alvo"], row["ativo_alvo"])

    linhas.append({
        "Indicador": indicador_nome,
        "Surpresa": "Positiva",
        "Ativo-alvo / Direção": f"{row['direcao_se_surpresa_positiva'].capitalize()} {ativo_nome}"
    })
    linhas.append({
        "Indicador": indicador_nome,
        "Surpresa": "Negativa",
        "Ativo-alvo / Direção": f"{row['direcao_se_surpresa_negativa'].capitalize()} {ativo_nome}"
    })

tabela_playbooks = pd.DataFrame(linhas)
print(tabela_playbooks.to_string(index=False))

     Indicador Surpresa                  Ativo-alvo / Direção
     CPI (EUA) Positiva   Vender ETF de índice de ações (SPY)
     CPI (EUA) Negativa  Comprar ETF de índice de ações (SPY)
 IPCA (Brasil) Positiva  Vender ETF da bolsa brasileira (EWZ)
 IPCA (Brasil) Negativa Comprar ETF da bolsa brasileira (EWZ)
Selic (Brasil) Positiva                 Vender câmbio USD/BRL
Selic (Brasil) Negativa                Comprar câmbio USD/BRL
 Payroll (EUA) Positiva   Vender ETF de índice de ações (SPY)
 Payroll (EUA) Negativa  Comprar ETF de índice de ações (SPY)


In [8]:
playbooks = pd.read_csv("../data/playbooks.csv")

operacoes = operacoes.merge(playbooks[["indicador", "ativo_alvo"]], on="indicador", how="left")

print(operacoes.columns.tolist())

['indicador', 'data', 'actual', 'forecast', 'diferenca', 'surpresa_zscore', 'atencao_bruta', 'IAN', 'ICE', 'limiar_ian_indicador', 'opera', 'direcao', 'tamanho_posicao', 'ativo_alvo']


In [9]:
tabela_operacoes = operacoes.copy()

tabela_operacoes["Indicador"] = tabela_operacoes["indicador"].map(nomes_indicadores)
tabela_operacoes["Ativo"] = tabela_operacoes["ativo_alvo"].map(nomes_ativos)
tabela_operacoes["Surpresa"] = tabela_operacoes["surpresa_zscore"].apply(lambda x: "Positiva" if x > 0 else "Negativa")
tabela_operacoes["Direção"] = tabela_operacoes["direcao"].str.capitalize()

tabela_final = tabela_operacoes[["data", "Indicador", "Surpresa", "Ativo", "Direção"]].rename(
    columns={"data": "Data"}
)
tabela_final = tabela_final.sort_values("Data")

print(tabela_final.to_string(index=False))

               Data      Indicador Surpresa                         Ativo Direção
         2023-04-12      CPI (EUA) Negativa  ETF de índice de ações (SPY) Comprar
2023-08-03 00:00:00 Selic (Brasil) Negativa                câmbio USD/BRL Comprar
         2024-04-10      CPI (EUA) Positiva  ETF de índice de ações (SPY)  Vender
2024-12-12 00:00:00 Selic (Brasil) Positiva                câmbio USD/BRL  Vender
2025-02-11 00:00:00  IPCA (Brasil) Negativa ETF da bolsa brasileira (EWZ) Comprar
2025-03-12 00:00:00  IPCA (Brasil) Positiva ETF da bolsa brasileira (EWZ)  Vender
2025-04-11 00:00:00  IPCA (Brasil) Positiva ETF da bolsa brasileira (EWZ)  Vender
         2026-02-13      CPI (EUA) Negativa  ETF de índice de ações (SPY) Comprar
         2026-03-06  Payroll (EUA) Negativa  ETF de índice de ações (SPY) Comprar
         2026-04-03  Payroll (EUA) Positiva  ETF de índice de ações (SPY)  Vender
         2026-04-10      CPI (EUA) Negativa  ETF de índice de ações (SPY) Comprar
         2026-06

In [10]:
print(eventos.groupby("indicador")["opera"].sum())

indicador
CPI_EUA        5
IPCA_BR        3
Payroll_EUA    3
Selic_BR       2
Name: opera, dtype: int64
